# Train baseline trên Colab GPU

Ở máy: chạy `python -m scripts.dong_goi_colab`, upload `experiments/source.zip` qua Files của Colab, chọn GPU. Notebook dùng gói mã đã upload, không phụ thuộc việc đã push GitHub hay chưa.


In [ ]:
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from uuid import uuid4

import google.colab  # noqa: F401 - Kiểm tra đang chạy trong Colab.
import torch

assert torch.cuda.is_available(), "Cần bật GPU T4."

PROJECT = Path("/content") / ("fashion_baseline_" + uuid4().hex[:8])
PROJECT.mkdir()

with zipfile.ZipFile("/content/source.zip") as source:
    source.extractall(PROJECT)

RUNS = PROJECT / "experiments/runs"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Kiểm tra thư viện

Kiểm tra CUDA và thư viện của phiên Colab.


In [ ]:
import matplotlib  # noqa: F401 - Kiểm tra thư viện có sẵn.
import numpy  # noqa: F401 - Kiểm tra thư viện có sẵn.
import pandas
import sklearn  # noqa: F401 - Kiểm tra thư viện có sẵn.
import torchvision  # noqa: F401 - Kiểm tra thư viện có sẵn.

print("Thư viện sẵn sàng. CUDA:", torch.cuda.is_available())

## 3. Train đầy đủ MLP rồi CNN

Train 5 epoch mỗi mô hình với Adam 0.001, batch 128. Lưu checkpoint có validation loss thấp nhất.


In [ ]:
before = set(RUNS.glob("*"))

with subprocess.Popen(
    [sys.executable, "-u", "-m", "training.huan_luyen"],
    cwd=PROJECT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    assert process.wait() == 0, "Xem lỗi ở trên."

created = set(RUNS.glob("*")) - before
assert len(created) == 1
RUN = created.pop()
assert json.loads((RUN / "STATUS.json").read_text())["stage"] == "completed"
print("TRAIN_COMPLETE:", RUN)

## 4. Đọc kết quả validation

So sánh loss và accuracy của từng epoch trên validation.


In [ ]:
from IPython.display import Image, display

for name in ["MLP", "CNN"]:
    print(name, json.loads((RUN / "metrics" / f"{name}_training.json").read_text()))
    display(pandas.read_csv(RUN / "metrics" / f"{name}_history.csv"))
    display(Image(filename=str(RUN / "figures" / f"{name}_learning_curve.png")))

## 5. Lưu kết quả về máy

Tải ZIP để giữ trọng số, cấu hình và lịch sử huấn luyện.


In [ ]:
from google.colab import files

archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
print("Kết quả:", archive)
files.download(archive)